In [ ]:
import torch
from constant import DATA_DIR, MIMIC_3_DIR,MIMIC_4_9_DIR,MIMIC_4_10_DIR
import pandas as pd
import numpy as np
from collections import defaultdict
import warnings
import json
import pickle
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

pathmimic = 'mimic3' # mimic4_9, mimic4_10

In [ ]:
def ind2c_freq_cooccurrencematrix(train_path):

    # Initialize frequency dictionary and code set
    freq = defaultdict(int)
    codes = set()
    samples = []
    
    # Iterate through data splits
    for split in ['train', 'dev', 'test']:
        temp_codes = set()
        temp_samples = []
        print(f"Processing split: {split}")
        df = pd.read_csv(train_path.replace('train', split))
        print(f"Opened file: {train_path.replace('train', split)}")
        for index, row in df.iterrows():

            sample_codes = set()
            if isinstance(row['LABELS'], str):
                for code in row['LABELS'].split(';'):

                    # code = code.strip().strip("'")
                    if len(code) == 0:
                        continue
                    freq[code] += 1
                    codes.add(code)
                    temp_codes.add(code)
                    sample_codes.add(code)
            else:
                print(f"skip invalid codes, index {index}: {row['LABELS']}")
            temp_samples.append(sample_codes)
            samples.append(sample_codes)  # Store the codes for the current sample
        print('all code in {}'.format(split), len(temp_codes))
        print('avg code in {}'.format(split), sum(len(c) for c in temp_samples)/len(temp_samples))
    # Create a code to index mapping, sorted by codes name
    ind2c = {i: c for i, c in enumerate(sorted(codes))}
    c2ind = {c: i for i, c in ind2c.items()}

    # Initialize co-occurrence matrix
    num_codes = len(codes)
    cooccurrence_matrix = np.zeros((num_codes, num_codes), dtype=int)

    # Populate the co-occurrence matrix
    for sample in samples:
        indices = [c2ind[code] for code in sample]
        for i in range(len(indices)):
            for j in range(i + 1, len(indices)):  # To avoid double counting
                cooccurrence_matrix[indices[i], indices[j]] = 1
                cooccurrence_matrix[indices[j], indices[i]] = 1
                
        # Ensure diagonal elements are set to 1 (code with itself)
        for index in indices:
            cooccurrence_matrix[index, index] = 1
    freq = dict(freq)
    print(f"Co-occurrence matrix shape: {cooccurrence_matrix.shape}")
    return ind2c, freq, cooccurrence_matrix
ind2c, freq, cooccurrence_matrix= ind2c_freq_cooccurrencematrix('{}/mimic3_train.csv'.format(MIMIC_3_DIR))


In [ ]:
# Split codes based on frequency threshold of 10
rare_codes = [code for code in freq if freq[code] <= 10]
common_codes = [code for code in freq if freq[code] > 10]
print(f"Number of rare codes (freq ≤ 10): {len(rare_codes)}")
print(f"Number of common codes (freq > 10): {len(common_codes)}")

In [ ]:
def calculate_condition_probabilities(rare_codes, common_codes, train_path):

    # Initialize frequency dictionary for rare codes, top codes, and co-occurrence matrix
    rare_freq = defaultdict(int)
    common_freq = defaultdict(int)
    samples = []
    
    with open('./icd_mimic3.json', 'r') as f:
        icd_mimic3_rank = json.load(f)
    listicd = list(icd_mimic3_rank.keys())

    # Iterate through data splits (train, dev, test)
    for split in ['train']:
        print(f"Processing split: {split}")
        df = pd.read_csv(train_path.replace('train', split))
        print(f"Opening file: {train_path.replace('train', split)}")
        for index, row in df.iterrows():
            sample_rare_codes = set()
            sample_common_codes = set()
            # Assuming the codes are in row[2]
            if isinstance(row['LABELS'], str):
                for code in row['LABELS'].split(';'):
                    if len(code) == 0:
                        continue
                    
                    # Check if code is in the rare or top codes set
                    if code in rare_codes:
                        sample_rare_codes.add(code)
                        rare_freq[code] += 1
                    if code in common_codes:
                        sample_common_codes.add(code)
                        common_freq[code] += 1
            else:
                print(f"skip invalid codes, index {index}: {row['LABELS']}")
            samples.append((sample_rare_codes, sample_common_codes))  # Store the sets of codes for the current sample

    # Create a rare-to-index mapping for easy lookup, sorted by codes name
    rare2ind = {rare: i for i, rare in enumerate(sorted(rare_codes))}
    ind2rare = {i: rare for rare, i in rare2ind.items()}
    common2ind = {common: i for i, common in enumerate(sorted(common_codes))}
    ground_ind_rare = {i: listicd.index(rare) for rare, i in rare2ind.items()}
    ground_ind_common = {i: listicd.index(common) for common, i in common2ind.items()}
    # Initialize the co-occurrence matrix for rare codes given top codes
    prob_matrix = np.zeros((len(rare_codes), len(common_codes)), dtype=float)

    # Calculate conditional probabilities P(top | rare)
    for sample_rare_codes, sample_common_codes in samples:
        # For each pair of rare and top codes in the sample
        for rare in sample_rare_codes:
            for common in sample_common_codes:
                rare_idx = rare2ind[rare]
                common_idx = common2ind[common]
                prob_matrix[rare_idx, common_idx] += 1

    # Normalize the matrix to calculate probabilities
    for i in range(len(ind2rare)):
        rare_count = rare_freq[ind2rare[i]]
        if rare_count > 0:
            prob_matrix[rare2ind[ind2rare[i]], :] /= rare_count  # Normalize by the number of occurrences of each rare code

    # Print the probability matrix
    print("Conditional Probability Matrix (rare code -> common code):")
    print(prob_matrix)
    print("rare_freq:")
    print(rare_freq)
    print("common_freq:")
    print(common_freq)

    np.save('../model_data/{}/{}_prob_matrix.npy'.format(pathmimic, pathmimic), prob_matrix)
    with open('../model_data/{}/{}_ground_ind_rare.pkl'.format(pathmimic, pathmimic), 'wb') as file:
        pickle.dump(ground_ind_rare, file)
    with open('../model_data/{}/{}_ground_ind_common.pkl'.format(pathmimic, pathmimic), 'wb') as file:
        pickle.dump(ground_ind_common, file)
    return prob_matrix, rare_freq, common_freq
train_path = ('{}/mimic3_train.csv'.format(MIMIC_3_DIR))

# Call the function with codes and train data path
prob_matrix, rare_freq, common_freq = calculate_condition_probabilities(rare_codes, common_codes, train_path)

In [ ]:
# Binning Algorithm for Comorbidity Encoding
adj_matrix = torch.ceil(torch.tensor(prob_matrix))
degree_list = torch.tensor(list(adj_matrix.sum(dim=1)) + list(adj_matrix.sum(dim=0)))

pickle.dump(adj_matrix, open('../model_data/{}/{}_adj_matrix.pkl'.format(pathmimic, pathmimic), 'wb'))

prob_np = prob_matrix.cpu().numpy() if torch.is_tensor(prob_matrix) else prob_matrix

# separate zero and non-zero values
non_zero = prob_np[prob_np != 0]

# generate equal CDF bins
num_bins = 10 # need 10 bins
cdf_quantiles = np.linspace(0, 1, num_bins + 1)
non_zero_bins = np.quantile(non_zero, cdf_quantiles)

# merge zero boundaries and convert to tensor
combined_bins = np.unique(np.concatenate([[0.0], non_zero_bins]))
print(combined_bins)
c_bins = torch.from_numpy(combined_bins).float()

# ensure bins are monotonically increasing
c_bins, _ = torch.sort(c_bins)

# calculate bin indices
prob_tensor = torch.tensor(prob_matrix) if not torch.is_tensor(prob_matrix) else prob_matrix
c_indices = torch.bucketize(prob_tensor, c_bins, right=True) - 1

# handle prob == 1 for total bin number
c_indices[prob_tensor == 1.0] = 10

print("c_indices",c_indices)
pickle.dump(c_indices, open('../model_data/{}/{}_c_indices.pkl'.format(pathmimic, pathmimic), 'wb'))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, LogFormatterSciNotation

def plot_freq_distribution(freq):
    
    plt.figure(figsize=(8, 6),dpi=300)
    sorted_freq = sorted(freq.values(), reverse=True)
    x = range(1, len(sorted_freq) + 1)

    plt.plot(x, sorted_freq, color = 'blue', label='MIMIC-III-ICD9 Code Frequency',lw=3)
    plt.xlabel('Sorted Code ID', fontsize=14)
    plt.ylabel('Code Frequency (log)', fontsize=14)
    plt.yscale('log')

    ax = plt.gca()
    ax.yaxis.set_major_locator(LogLocator(base=10))
    ax.yaxis.set_major_formatter(LogFormatterSciNotation(base=10))

    target_x = None
    for i, f in enumerate(sorted_freq, start=1):
        if f == 10:
            target_x = i
            break
    if target_x is not None:

        plt.axvline(x=target_x, color='red', linestyle='--', label='Frequency = 10',lw=2)
    plt.axhline(y=10, color='grey', linestyle='--',lw=1)
    plt.axhline(y=100, color='grey', linestyle='--',lw=1)
    plt.axhline(y=1000, color='grey', linestyle='--',lw=1)
    plt.axhline(y=10000, color='grey', linestyle='--',lw=1)
    plt.legend(loc='upper right', fontsize=12)
    ax.tick_params(axis='both', which='major', labelsize=10)
    plt.show()

plot_freq_distribution(freq)